# M2 RepeatShare 인기도 통제 역사적 개발 백테스트

이미 확인한 validation/test/holdout을 다시 보지 않고, Dunnhumby 1~683일로 학습해 684~690일 신규상품 추천을 평가합니다.

비교 모형은 `M1@64`, 원본 `RepeatShare` M2, 인기도의 선형 성분을 통제한 `RepeatShare` M2입니다. 두 M2 사이에는 아이템 N 입력 첫 열 하나만 다릅니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '6ec8535b7fafecadb2b0c6991231a9a8c1a7bc92'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_repeatshare_backtest import (
    configure_repeatshare_backtest,
    preflight_summary,
    run_repeatshare_backtest,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_repeatshare_backtest(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    )
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_repeatshare_backtest(cfg)

In [ ]:
from IPython.display import display

print('절대지표:')
display(result_df.sort_values('model_id'))

comparison = result_df.attrs['comparison']
core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('핵심 비교표:')
display(
    comparison[comparison['metric'].isin(core_metrics)]
    .sort_values(['model_id', 'reference', 'metric'])
)
print('결과 파일:', result_df.attrs['result_paths'])